# On-Policy Distillation: When Self-Generated Data Wins

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/on_policy_distillation.ipynb)

Companion notebook to the [blog post](https://www.sesen.ai/blog/on-policy-distillation).

We build **off-policy (SeqKD)** and **on-policy (GKD)** distillation from scratch and test
them on a task where one wrong token is unrecoverable: generating balanced brackets (the
Dyck-1 language). The teacher is an **oracle** that knows the exact valid-completion
distribution from any prefix, so the only variable is which states the student trains on.

The honest finding: on this small, fully-supervised task the two methods are
indistinguishable. The advantage of on-policy distillation is a *distribution-shift and
scale* effect, and we explain exactly when it appears.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

## The task and the oracle teacher

A valid Dyck-1 string keeps its running depth non-negative and ends at zero. We count, with
the ballot numbers, how many valid completions go through `(` vs `)` from any `(depth,
remaining)` state. That gives an exact per-token target distribution for **any** prefix,
including the broken ones a student invents.

In [ ]:
OPEN, CLOSE, BOS, VOCAB = 0, 1, 2, 3
L = 12                 # bracket pairs
T = 2 * L              # symbols per string
SEQ = T + 1            # with BOS

# Ncomp[d][r] = number of length-r continuations that stay >= 0 and end at depth 0
Ncomp = [[0] * (T + 1) for _ in range(T + 2)]
for d in range(T + 1):
    Ncomp[d][0] = 1 if d == 0 else 0
for r in range(1, T + 1):
    for d in range(T + 1):
        opn = Ncomp[d + 1][r - 1] if d + 1 <= T else 0
        cls = Ncomp[d - 1][r - 1] if d - 1 >= 0 else 0
        Ncomp[d][r] = opn + cls

# ORACLE[d, r] = (P(open), P(close)) target distribution
ORACLE = torch.zeros(T + 2, T + 1, VOCAB)
for d in range(T + 2):
    for r in range(1, T + 1):
        tot = Ncomp[d][r] if d <= T else 0
        if tot:
            po = (Ncomp[d + 1][r - 1] if d + 1 <= T else 0) / tot
        else:
            po = 0.0 if d > 0 else 1.0   # off-manifold: close to recover
        ORACLE[d, r, OPEN] = po
        ORACLE[d, r, CLOSE] = 1 - po

In [ ]:
def sample_valid(n, gen):
    # Sample n uniform Dyck strings by following the oracle.
    out = torch.full((n, SEQ), BOS, dtype=torch.long)
    depth = torch.zeros(n, dtype=torch.long)
    for t in range(T):
        po = ORACLE[depth, T - t, OPEN]
        tok = torch.where(torch.rand(n, generator=gen) < po, OPEN, CLOSE)
        out[:, t + 1] = tok
        depth = depth + torch.where(tok == OPEN, 1, -1)
    return out

def depths_and_remaining(seq):
    # Depth before each emitted token and the steps remaining, for oracle lookup.
    B, dev = seq.size(0), seq.device
    step = torch.where(seq[:, 1:] == OPEN, 1, -1)
    depth_before = torch.zeros(B, T, dtype=torch.long, device=dev)
    d = torch.zeros(B, dtype=torch.long, device=dev)
    for t in range(T):
        depth_before[:, t] = d
        d = (d + step[:, t]).clamp(0, T)
    remaining = torch.arange(T, 0, -1, device=dev)
    return depth_before, remaining.expand(B, T)

def oracle_targets(seq):
    d, r = depths_and_remaining(seq)
    return ORACLE.to(seq.device)[d, r]

g_train = torch.Generator().manual_seed(1)
g_test = torch.Generator().manual_seed(99)
TRAIN = sample_valid(12000, g_train)
print("train strings:", TRAIN.shape, "| all balanced:",
      bool(((TRAIN[:, 1:] == OPEN).sum(1) == L).all()))

## The student: a deliberately tiny Transformer

A single layer with `d_model=16`, about 3,800 parameters. Small enough that it has to make
trade-offs, which is the regime where the choice of training data could matter.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, d_model=16, n_layer=1, n_head=2):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, d_model)
        self.pos = nn.Embedding(SEQ, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_head, 4 * d_model,
                                           batch_first=True, activation="gelu", dropout=0.0)
        self.blocks = nn.TransformerEncoder(layer, n_layer)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, VOCAB)
        self.register_buffer("mask", torch.triu(torch.full((SEQ, SEQ), float("-inf")), 1))

    def forward(self, x):
        Tn = x.size(1)
        h = self.tok(x) + self.pos(torch.arange(Tn, device=x.device))
        h = self.blocks(h, mask=self.mask[:Tn, :Tn])
        return self.head(self.ln(h))

def pred_logits(model, seq):
    return model(seq)[:, :T]      # predict tokens 1..T

print("student params:", sum(p.numel() for p in TinyGPT().parameters()))

## Measuring success: free-running validity

We sample whole strings from the student and check whether they are balanced. This is the
inference-time behaviour that exposure bias would damage.

In [ ]:
@torch.no_grad()
def validity(model, n=2000, temp=1.0, seed=7):
    model.eval()
    gen = torch.Generator().manual_seed(seed)
    seq = torch.full((n, 1), BOS, dtype=torch.long, device=DEVICE)
    depth = torch.zeros(n, dtype=torch.long, device=DEVICE)
    alive = torch.ones(n, dtype=torch.bool, device=DEVICE)
    for _ in range(T):
        logits = model(seq)[:, -1, :2] / temp
        p = F.softmax(logits, -1)
        u = torch.rand(n, generator=gen).to(DEVICE)
        tok = torch.where(u < p[:, OPEN], OPEN, CLOSE)
        alive = alive & ~((tok == CLOSE) & (depth == 0))
        depth = (depth + torch.where(tok == OPEN, 1, -1)).clamp(min=0)
        seq = torch.cat([seq, tok.unsqueeze(1)], 1)
    return (alive & (depth == 0)).float().mean().item()

## The two distillation rules

Both minimise the **same** forward KL to the **same** oracle. They differ only in which
sequences the student is trained on: ground-truth strings (off-policy) or the student's own
sampled rollouts (on-policy).

In [ ]:
def run_phase(m, mode, epochs, lr, bs=512, clip=1.0, label=""):
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    n, hist, best, best_state = TRAIN.size(0), [], -1.0, None
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            seq = TRAIN[perm[i:i + bs]].to(DEVICE)
            if mode == "on":                       # student rolls out its own sequence
                with torch.no_grad():
                    roll = seq[:, :1].clone()
                    for _ in range(T):
                        p = F.softmax(m(roll)[:, -1, :2], -1)
                        roll = torch.cat([roll, torch.multinomial(p, 1)], 1)
                    seq = roll
            with torch.no_grad():
                target = oracle_targets(seq)
            logp = F.log_softmax(pred_logits(m, seq), -1)
            loss = F.kl_div(logp, target, reduction="batchmean")
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), clip); opt.step()
        v = validity(m)
        hist.append(v)
        if v > best:                                # keep-best: on-policy can overtrain
            best, best_state = v, {k: x.detach().cpu().clone() for k, x in m.state_dict().items()}
        print(f"  {label} ep{ep} validity {v:.3f} (best {best:.3f})")
    m.load_state_dict(best_state)
    return hist

## Train both from an identical warm start

We warm the student with a few off-policy epochs (so its rollouts are not pure noise), then
split into an off-policy branch and an on-policy branch from the same checkpoint.

In [ ]:
def train_both(seed=0, warm=6, tail=18, lr=2e-3):
    torch.manual_seed(seed)
    base = TinyGPT().to(DEVICE)
    warm_hist = run_phase(base, "off", warm, lr, label="warm")
    state = {k: v.detach().cpu().clone() for k, v in base.state_dict().items()}
    off = TinyGPT().to(DEVICE); off.load_state_dict(state)
    off_hist = run_phase(off, "off", tail, lr, label="off ")
    on = TinyGPT().to(DEVICE); on.load_state_dict(state)
    on_hist = run_phase(on, "on", tail, lr, label="on  ")
    return off, on, warm_hist + off_hist, warm_hist + on_hist, len(warm_hist)

off_m, on_m, off_hist, on_hist, warm_len = train_both()
print(f"\nfinal greedy-ish validity: off {validity(off_m):.3f}  on {validity(on_m):.3f}")

In [ ]:
ep = range(1, len(off_hist) + 1)
plt.figure(figsize=(8, 4.5))
plt.axvspan(0, warm_len + 0.5, color="grey", alpha=0.08)
plt.axvline(warm_len + 0.5, color="grey", ls=":", lw=1)
plt.plot(ep, off_hist, color="#d1495b", marker="o", ms=3, label="Off-policy (SeqKD)")
plt.plot(ep, on_hist, color="#1f7a8c", marker="s", ms=3, label="On-policy (GKD)")
plt.xlabel("Distillation epoch"); plt.ylabel("Free-running validity")
plt.title("Both methods distil the teacher to high bracket validity")
plt.ylim(0, 1.02); plt.legend(frameon=False); plt.grid(alpha=0.25); plt.show()

## Robustness: push both off the greedy path

Both students are near-perfect greedily. Do they differ when forced off their comfortable
trajectory by hotter sampling? On this matched-distribution toy, no, the curves sit on top
of each other.

In [ ]:
temps = [1.0, 1.5, 2.0, 2.5, 3.0]
off_t = [validity(off_m, temp=t) for t in temps]
on_t = [validity(on_m, temp=t) for t in temps]
plt.figure(figsize=(8, 4.5))
plt.plot(temps, off_t, color="#d1495b", marker="o", ms=5, label="Off-policy (SeqKD)")
plt.plot(temps, on_t, color="#1f7a8c", marker="s", ms=5, label="On-policy (GKD)")
plt.xlabel("Sampling temperature"); plt.ylabel("Free-running validity")
plt.title("Off-policy holds up as well as on-policy on this clean task")
plt.ylim(0, 1.02); plt.legend(frameon=False); plt.grid(alpha=0.25); plt.show()
print("off:", [round(x, 2) for x in off_t])
print("on :", [round(x, 2) for x in on_t])

## When on-policy distillation actually wins

The two methods tie here because the oracle covers every state and the training distribution
already matches what the student generates: there is no coverage gap for on-policy to
exploit. On-policy distillation pays off when those conditions break:

1. **A real coverage gap** between the teacher's data and the student's inference states
   (long chain-of-thought, agentic rollouts, code).
2. **A student large enough** to optimise stably on its own noisy rollouts.
3. **Mode-seeking** (reverse KL) when the student cannot represent the teacher exactly.

All three hold at LLM-reasoning scale and none hold on a tiny, clean toy. See
[Agarwal et al. (2024) GKD](https://arxiv.org/abs/2306.13649),
[Gu et al. (2024) MiniLLM](https://arxiv.org/abs/2306.08543), and the
[Thinking Machines Lab (2025) post](https://thinkingmachines.ai/blog/on-policy-distillation).

## Exercises

1. **Make a coverage gap.** Train the off-policy branch only on shallow strings (cap the
   nesting depth), then test completion from deep prefixes. Does on-policy now pull ahead?
2. **Reverse KL.** Swap the forward KL for reverse KL `kl_div(teacher_logp, student_prob)`.
   Does it change the mode-seeking behaviour at high temperature?
3. **GKD mixture.** Mix off-policy and on-policy batches with a coefficient lambda and sweep
   it from 0 to 1.
4. **Harder task.** Increase `L` and shrink the student until off-policy starts to fail at
   inference. Where does the crossover happen?